In [1]:
import os
import qdrant_client
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core import QueryBundle
import asyncio
from llama_index.core.vector_stores.types import MetadataFilters, MetadataFilter, FilterOperator, VectorStoreQueryMode
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.agent.workflow import FunctionAgent




load_dotenv()

VLLM_API_BASE_URL = os.getenv("VLLM_API_BASE_URL")
COLLECTION_NAME = "WAMASRAGBASE"
QDRANT_URL = os.getenv("QDRANT_URL")



In [2]:
llm = OpenAILike(
    model='Qwen/Qwen2.5-32B-Instruct-AWQ',
    api_base=VLLM_API_BASE_URL, 
    api_key="null",
    is_chat_model=True,
    is_function_calling_model=True,    
    timeout=60.0,
    streaming=True,
    context_window=4096, # CHECK
    temperature=0,
)
embed_model = OpenAIEmbedding(
    api_base=VLLM_API_BASE_URL,
    model_name="BAAI/bge-m3",
    api_key="null",
)
Settings.embed_model = embed_model
Settings.llm = llm

client = qdrant_client.QdrantClient(url=QDRANT_URL)
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)


### prova agente

In [3]:
import asyncio
from llama_index.core.agent.workflow import ReActAgent


# Define a simple calculator tool
def multiply(a: float, b: float) -> float:
    """Useful for multiplying two numbers."""
    return a * b


# Create an agent workflow with our calculator tool
agent = ReActAgent(
    tools=[multiply],
    llm=llm,
    system_prompt="You are a helpful assistant that can multiply two numbers.",
    verbose=True,
)


async def main():
    # Run the agent
    response = await agent.run("quanto é 12.5 moltiplicato per 4?")
    print(str(response))


# Run the agent
await main()

12.5 moltiplicato per 4 è 50.0.


### fine prova agente

In [4]:
import requests

url = os.getenv("RERANKER_API_URL")

def reranka(nodes, query_text, top_n=5):

    nodinuovi = []
    noditesto = []


    if not nodes:
        return []


    for n in nodes:
        if hasattr(n, 'node'): 

            content = n.node.get_content()
        else:

            content = n.get_content()
        noditesto.append(content)

    payload = {
        "model": "BAAI/bge-reranker-v2-m3",
        "query": f"{query_text}",
        "documents": noditesto,
        "top_n": top_n
    }

    headers = {
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        
        
        if response.status_code != 200:
            print(f"Errore API Rerank ({response.status_code}): {response.text}")
            response.raise_for_status()
            
        results = response.json()
        
        
        for item in results.get("results", []):
            original_index = item["index"]
        
            nodinuovi.append(nodes[original_index])

    except Exception as e:
        print(f" Errore durante il reranking: {e}")
        
        return nodes[:top_n]

    return nodinuovi


In [6]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)


def get_context_from_knowledge_base(query_text, user_role="admin", hydeaugmented=None, queryaugmentation=None):
    print(f"DEBUG: Ricerca per: '{query_text}' - Ruolo: '{user_role}'")

    filters = MetadataFilters(
        filters=[
            MetadataFilter(
                key="groups",  
                value=user_role,    
                operator=FilterOperator.EQ 
            )
        ]
    )

    topk = 20
    
    retriever_dense = index.as_retriever(
        vector_store_query_mode=VectorStoreQueryMode.DEFAULT, 
        similarity_top_k=topk,
        filters=filters if user_role != "admin" else None
    )
    retriever_sparse = index.as_retriever(
        vector_store_query_mode=VectorStoreQueryMode.SPARSE,
        sparse_top_k=topk,
        filters=filters if user_role != "admin" else None
    )

    retriever = QueryFusionRetriever(
        retrievers=[retriever_dense, retriever_sparse],
        similarity_top_k=topk,
        num_queries=1,          
        mode="simple",                                       
        use_async=False
    )

    query_bundle = QueryBundle(
        query_str=query_text,
        custom_embedding_strs=hydeaugmented if hydeaugmented else (queryaugmentation if queryaugmentation else None)
    )

    nodes = retriever.retrieve(query_bundle)

    if not nodes:
        return "Nessuna informazione rilevante trovata nel database.", " "

    # Reranking
    nodes = reranka(nodes, query_text, top_n=5)
    
    context_parts = []
    docs_dict = {}
    
    for node_ws in nodes:
        filename = node_ws.node.metadata.get('origin_filename', 'Unknown File')
        page = node_ws.node.metadata.get('pages', 'N/A')
        content = node_ws.node.get_content().strip()
        context_parts.append(f"--- Documento: {filename} (Pag. {page}) ---\n{content}\n")
        
        if filename not in docs_dict:
            docs_dict[filename] = []
        
        
        if isinstance(page, list):
            docs_dict[filename].extend(page)
        else:
            docs_dict[filename].append(page)
    
    full_context = "\n".join(context_parts)
    
    return full_context


#Tool Retrieval
def search_knowledge_base(query: str) -> str:
    
    CURRENT_USER_ROLE = "admin" 
    
    try:
        context_str = get_context_from_knowledge_base(query, user_role=CURRENT_USER_ROLE)
        print(f"DEBUG: Contesto recuperato:\n{context_str}")
        return context_str
        
    except Exception as e:
        return f"Errore durante la ricerca: {str(e)}"

retrieval_tool = FunctionTool.from_defaults(fn=search_knowledge_base)


#memory = ChatMemoryBuffer.from_defaults(token_limit=1000)

agent = FunctionAgent(
    tools=[retrieval_tool],
    llm=llm,
    system_prompt="""Sei un assistente intelligente. 
    Hai accesso a un database di documenti per le domande informative.
    Usa sempre lo strumento più appropriato per la domanda..""",
    #memory=memory,
    verbose=True,
    max_iterations=10
)



async def main():

    print("Retrieval RAG")
    
    query_rag = "come si crea una udc?" 
    response_rag = await agent.run(query_rag)
    print(f"Risposta: {response_rag}")

if __name__ == "__main__":
    
    await main()

Retrieval RAG
DEBUG: Ricerca per: 'come si crea una UDC' - Ruolo: 'admin'
DEBUG: Contesto recuperato:
--- Documento: PUB_Creazione_nuove_UDC.pdf (Pag. [2]) ---
3. Come creare nuove unità di carico
Per creare  una nuova UDC si parte dalla pagina SM023 su WAMAS. Si clicca tasto destro in un'area vuota e cliccare 'Nuova'
[IMAGE] L'immagine mostra l'interfaccia del modulo SM023 per la gestione delle unità di carico in un sistema ERP, con un campo di ricerca e filtri per ID UDC, base di carico, magazzino e posto di stoccaggio. È evidenziato il menu contestuale "Nuova..." che offre opzioni per creare nuove unità di carico, modificare, annullare ordini di trasporto, stampare etichette o eseguire prelievi spontanei. La sezione "Risultato di ricerca" è vuota, indicando che nessun record è stato trovato con i criteri attuali. [/IMAGE]

--- Documento: PUB_Creazione_nuove_UDC.pdf (Pag. [2]) ---
3. Come creare nuove unità di carico
Si aprirà così la finestra SM028 ; nella vista 'Unità di carico' an